# 02 — Tiny Aya CircuitKIT EAP-IG discovery

This notebook runs only after `01_prepare_circuitkit_data.ipynb` has written a passing `behavior_validation.json` and the repository’s adjudication workflow has exported human-approved, seed-unique analysis pairs. It discovers **separate English and Japanese circuits** with an independently sampled, reproducible cohort in each language. It does not pool the languages into one attribution objective. No automatic guard labels or provisional rows are accepted.

CircuitKIT 0.2.2 / TransformerLens 3.8.0 is pinned to commit `e3501bc72b349963db76df324f5dbd41ced2a02f`. Apply the repository’s position-map patch and verify the imported source path. The patch handles only explicit exact-prefix and identical-continuation maps; divergent framing positions remain unmapped.

Outputs are private runtime artifacts. Copy them to the project’s dedicated private Drive output folder and verify checksums before stopping the runtime. Attribution ranks nominate candidates; they do not establish causality. CircuitKIT graph edges are visualization layout, not measured causal connections.


In [ ]:
# Install pinned dependencies, then build a disposable patched CircuitKIT checkout.
# The patch and `eval/track2_extension` project files must be uploaded under /content/lexsi.
import importlib.metadata as _md
import os as _os
from pathlib import Path as _Path
import subprocess as _subprocess
import sys as _sys

_project_root = _Path(_os.environ.get("LEXSI_ROOT", "/content/lexsi"))
_patch_file = _project_root / "docs/lexsi/patches/circuitkit_eap_position_maps_e3501bc.patch"
if not _patch_file.is_file():
    raise FileNotFoundError(f"Upload the repository patch to {_patch_file}")
if not (_project_root / "eval/track2_extension/circuitkit_data.py").is_file():
    raise FileNotFoundError(f"Upload eval/track2_extension under {_project_root}")
_torch_before = _md.version("torch")
_subprocess.check_call([_sys.executable, "-m", "pip", "install", "circuitkit==0.2.2", "transformers==5.15.0", "transformer-lens==3.8.0"])
_torch_after = _md.version("torch")
if _torch_after != _torch_before:
    raise RuntimeError(f"Dependency install changed PyTorch {_torch_before} -> {_torch_after}; restore the Colab CUDA runtime before continuing.")
_expected = {"circuitkit": "0.2.2", "transformers": "5.15.0", "transformer-lens": "3.8.0"}
_installed = {name: _md.version(name) for name in _expected}
if _installed != _expected:
    raise RuntimeError(f"Unexpected dependency versions: {_installed}")
_target = "e3501bc72b349963db76df324f5dbd41ced2a02f"
_source = _Path(_os.environ.get("CIRCUITKIT_PATCHED_SOURCE", str(_project_root / "_CircuitKIT_e3501bc")))
if not _source.exists():
    _subprocess.check_call(["git", "clone", "https://github.com/Lexsi-Labs/CircuitKIT.git", str(_source)])
    _subprocess.check_call(["git", "-C", str(_source), "checkout", _target])
_commit = _subprocess.check_output(["git", "-C", str(_source), "rev-parse", "HEAD"], text=True).strip()
if _commit != _target:
    raise RuntimeError(f"Refusing to change existing CircuitKIT checkout at {_source}: expected {_target}, found {_commit}")
_check = _subprocess.run(["git", "-C", str(_source), "apply", "--check", str(_patch_file)], capture_output=True, text=True)
if _check.returncode == 0:
    _subprocess.check_call(["git", "-C", str(_source), "apply", str(_patch_file)])
else:
    _reverse = _subprocess.run(["git", "-C", str(_source), "apply", "--reverse", "--check", str(_patch_file)], capture_output=True, text=True)
    if _reverse.returncode != 0:
        raise RuntimeError(f"Pinned CircuitKIT patch is neither cleanly applicable nor already applied: {_check.stderr} {_reverse.stderr}")
_os.environ["CIRCUITKIT_PATCHED_SOURCE"] = str(_source)
print({"torch": _torch_after, **_installed, "circuitkit_commit": _commit, "patched_source": str(_source)})


In [ ]:
import hashlib, importlib.metadata as metadata, json, os, platform, subprocess, sys
from pathlib import Path
import torch

ROOT = Path(os.environ.get("LEXSI_ROOT", "/content/lexsi"))
PHASE1 = ROOT / "phase1" / "behavior_validation.json"
APPROVED = ROOT / "circuitkit_pairs.jsonl"
PATCHED_CIRCUITKIT = Path(os.environ.get("CIRCUITKIT_PATCHED_SOURCE", "/content/CircuitKIT"))
RUN = ROOT / "circuitkit_discovery"
RUN.mkdir(parents=True, exist_ok=True)
MODEL = "CohereLabs/tiny-aya-water"
REVISION = "41a99216d12b3ef4881df762934186e5e4d2bbc1"
TASK = "lexsi_refusal_compliance_analysis"
ARTIFACT = RUN / "analysis_eap_ig_circuit.pt"

assert torch.cuda.is_available(), "A real Colab GPU runtime is required."
assert PHASE1.is_file(), "Run notebook 01 first; behavior_validation.json is missing."
phase1 = json.loads(PHASE1.read_text())
assert phase1.get("passed") is True, "Notebook 01 metric validation did not pass."
assert APPROVED.is_file() and APPROVED.stat().st_size, "No human-approved CircuitKIT export exists."
assert PATCHED_CIRCUITKIT.exists(), "Apply the pinned CircuitKIT patch checkout first."
source_commit = subprocess.check_output(["git", "-C", str(PATCHED_CIRCUITKIT), "rev-parse", "HEAD"], text=True).strip()
assert source_commit == "e3501bc72b349963db76df324f5dbd41ced2a02f", source_commit
sys.path.insert(0, str(PATCHED_CIRCUITKIT / "src"))
sys.path.insert(0, str(ROOT))
import circuitkit as ck
from circuitkit.tasks import register_task
from circuitkit.backends.eap import eap_utils
assert "position_map" in eap_utils.collate_EAP_with_position_maps.__name__, "Position-map patch is not imported."
from eval.track2_extension.circuitkit_data import RefusalComplianceTaskSpec

versions = {p: metadata.version(p) for p in ("torch", "transformers", "circuitkit", "transformer-lens")}
assert versions["circuitkit"] == "0.2.2" and versions["transformer-lens"] == "3.8.0", versions
print({"versions": versions, "gpu": torch.cuda.get_device_name(0), "torch_cuda": torch.version.cuda})


In [ ]:
# Validate the complete export and draw a deterministic, balanced sample independently by language.
raw = APPROVED.read_bytes()
pairs = [json.loads(line) for line in raw.splitlines() if line.strip()]
if not pairs:
    raise ValueError("Approved export contains no pairs.")
from eval.track2_extension.discovery import select_discovery_pairs_by_language
pairs_by_language = select_discovery_pairs_by_language(
    pairs, seed=20261112, sample_size_per_language=16, minimum_per_language=8,
)
print({language: len(rows) for language, rows in pairs_by_language.items()})

from circuitkit.tasks.registry import _TASKS
import gc
import circuitkit.api as ck_api
from circuitkit import Circuit

# Preserve raw signed edge scores before CircuitKIT's public node aggregation.
_original_attribute_node = ck_api.attribute_node
active_signed_edge_path = None
def _capture_signed_edge_scores(*args, **kwargs):
    graph = args[1] if len(args) > 1 else kwargs["graph"]
    result_inner = _original_attribute_node(*args, **kwargs)
    edge_rows = [
        {"edge": name, "signed_score": float(edge.score.detach().cpu()), "in_graph": bool(edge.in_graph)}
        for name, edge in graph.edges.items()
    ]
    active_signed_edge_path.write_text(json.dumps(edge_rows, indent=2) + "\n")
    return result_inner

metadata_by_language = {}
ck_api.attribute_node = _capture_signed_edge_scores
try:
    for language in ("en", "ja"):
        language_pairs = pairs_by_language[language]
        task_name = f"{TASK}_{language}"
        _TASKS.pop(task_name, None)
        register_task(RefusalComplianceTaskSpec(pairs=language_pairs, name=task_name))

        # Load the same immutable HF checkpoint for each independent language run.
        model = ck.load_model(
            MODEL, revision=REVISION, dtype="float32", device="cuda", token=True, algorithm="eap-ig"
        ).eval()
        assert model.cfg.n_layers == 36 and model.cfg.d_model == 2048
        model.cfg.ungroup_grouped_query_attention = True
        torch.cuda.reset_peak_memory_stats()
        artifact = RUN / f"analysis_{language}_eap_ig_circuit.pt"
        active_signed_edge_path = RUN / f"raw_signed_edge_attributions_{language}.json"
        config = {
            "model": {"name": MODEL, "precision": "float32", "device": "cuda"},
            "discovery": {
                "algorithm": "eap-ig", "task": task_name, "level": "node", "method": "EAP-IG-inputs",
                "batch_size": 2, "ig_steps": 3, "seed": 20261112, "verbose": True,
                "max_seq_len": 2048, "data_params": {"num_examples": 2 * len(language_pairs), "seed": 20261112},
            },
            "pruning": {"target_sparsity": 0.10, "scope": "both", "intervention": "patching"},
            "output_path": str(artifact),
        }
        ck.discover_circuit(config, _model=model)
        score_pt = artifact.with_name(artifact.stem + "_scores.pt")
        score_json = artifact.with_name(artifact.stem + "_scores.json")
        if not all(path.is_file() for path in (artifact, score_pt, score_json, active_signed_edge_path)):
            raise FileNotFoundError(f"CircuitKIT did not write all expected {language} artifacts")

        circuit = Circuit.from_artifact(str(artifact))
        ck.visualize_circuit(circuit, mode="graph", output=str(RUN / f"analysis_{language}_circuit.html"))
        scores = json.loads(score_json.read_text())
        node_scores = scores.get("node_scores", {})
        ranked = sorted(node_scores.items(), key=lambda item: abs(float(item[1])), reverse=True)
        (RUN / f"ranked_{language}_node_attributions.json").write_text(json.dumps(
            [{"component": name, "absolute_attribution": float(value)} for name, value in ranked], indent=2
        ) + "\n")
        seed_ids = sorted(pair["seed_id"] for pair in language_pairs)
        metadata_by_language[language] = {
            "language": language, "artifact": artifact.name, "score_json": score_json.name,
            "model": MODEL, "revision": REVISION, "algorithm": "eap-ig", "ig_steps": 3,
            "seed": 20261112, "sample_seed": 20261112, "analysis_pairs": len(language_pairs),
            "analysis_seed_ids": seed_ids,
            "analysis_seed_ids_sha256": hashlib.sha256("\n".join(seed_ids).encode()).hexdigest(),
            "selected_row_ids": sorted(pair["row_id"] for pair in language_pairs),
            "circuitkit_source_commit": source_commit,
            "approved_export_sha256": hashlib.sha256(raw).hexdigest(),
            "circuit_nodes": list(circuit.nodes), "n_scored_nodes": len(node_scores),
            "signed_edge_score_rows": len(json.loads(active_signed_edge_path.read_text())),
            "versions": versions, "gpu": torch.cuda.get_device_name(0),
            "peak_allocated_bytes": torch.cuda.max_memory_allocated(),
            "peak_reserved_bytes": torch.cuda.max_memory_reserved(),
            "score_semantics": "Node attribution is magnitude-only; signed edge scores are saved separately. Graph edges are layout only.",
            "interpretation": "Language-specific attribution candidates only; no causal claim.",
        }
        print(language, json.dumps({key: metadata_by_language[language][key] for key in (
            "analysis_pairs", "n_scored_nodes", "peak_allocated_bytes", "peak_reserved_bytes"
        )}, indent=2))
        del circuit, model
        gc.collect()
        torch.cuda.empty_cache()
finally:
    ck_api.attribute_node = _original_attribute_node

all_seed_ids = sorted({seed for data in metadata_by_language.values() for seed in data["analysis_seed_ids"]})
meta = {
    "model": MODEL, "revision": REVISION, "circuitkit_source_commit": source_commit,
    "selection": {"method": "independent seeded sample within language", "sample_seed": 20261112,
                  "maximum_per_language": 16, "minimum_per_language": 8},
    "languages": metadata_by_language, "analysis_seed_ids": all_seed_ids,
    "analysis_seed_ids_sha256": hashlib.sha256("\n".join(all_seed_ids).encode()).hexdigest(),
    "approved_export_sha256": hashlib.sha256(raw).hexdigest(), "versions": versions,
    "interpretation": "Separate EN/JA EAP-IG attribution candidates; no causal or cross-language transfer claim.",
}
(RUN / "discovery_metadata.json").write_text(json.dumps(meta, indent=2) + "\n")


In [ ]:
# Audit the saved outputs for both language-specific candidate circuits.
meta = json.loads((RUN / "discovery_metadata.json").read_text())
for language in ("en", "ja"):
    data = meta["languages"][language]
    artifact = RUN / data["artifact"]
    assert artifact.is_file() and artifact.with_name(artifact.stem + "_scores.pt").is_file()
    assert artifact.with_name(artifact.stem + "_scores.json").is_file()
    assert (RUN / f"analysis_{language}_circuit.html").is_file()
    assert (RUN / f"raw_signed_edge_attributions_{language}.json").is_file()
    assert (RUN / f"ranked_{language}_node_attributions.json").is_file()
    print(language, {key: data[key] for key in ("analysis_pairs", "n_scored_nodes", "signed_edge_score_rows")})
print("Separate EN/JA artifacts verified. Attribution remains candidate evidence only.")
